# 01 — Build cell-type prototypes

Reads scRNA-seq data and produces per-cell-type average gene expression profiles (prototypes).

**Inputs**
- `data/scrna_expression.csv` — raw scRNA-seq count matrix (cells × genes)
- `data/scrna_metadata.csv` — cell metadata with a `cell_type` column

**Outputs**
- `data/prototypes.csv` — mean expression per cell type (cell_types × genes)
- `data/prototypes_normalized.csv` — L2-normalised version for cosine similarity
- `data/shared_genes.txt` — genes present in both scRNA-seq and Xenium data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

## Config

In [ ]:
SCRNA_EXPR_PATH   = "../data/scrna_expression.csv"
SCRNA_META_PATH   = "../data/scrna_metadata.csv"
XENIUM_GENES_PATH = "../data/xenium_gene_panel.txt"  # one gene name per line

PROTOTYPES_PATH            = "../data/prototypes.csv"
PROTOTYPES_NORM_PATH       = "../data/prototypes_normalized.csv"
SHARED_GENES_PATH          = "../data/shared_genes.txt"

CELL_TYPE_COL = "cell_type"  # column name in metadata

## 1 · Load scRNA-seq data

In [ ]:
expr = pd.read_csv(SCRNA_EXPR_PATH, index_col=0)
meta = pd.read_csv(SCRNA_META_PATH, index_col=0)

print("Expression matrix:", expr.shape, "(cells × genes)")
print("Metadata:", meta.shape)
print("Cell types:", meta[CELL_TYPE_COL].unique())

## 2 · Find shared genes with Xenium panel

In [ ]:
with open(XENIUM_GENES_PATH) as f:
    xenium_genes = [line.strip() for line in f if line.strip()]

shared_genes = sorted(set(expr.columns) & set(xenium_genes))
print(f"scRNA-seq genes: {len(expr.columns)}")
print(f"Xenium panel genes: {len(xenium_genes)}")
print(f"Shared genes: {len(shared_genes)}")

expr = expr[shared_genes]

with open(SHARED_GENES_PATH, "w") as f:
    f.write("\n".join(shared_genes))
print(f"Saved shared genes → {SHARED_GENES_PATH}")

## 3 · Build prototypes by averaging per cell type

In [ ]:
# align expression matrix and metadata on shared cell indices
shared_cells = expr.index.intersection(meta.index)
expr = expr.loc[shared_cells]
meta = meta.loc[shared_cells]

prototypes = expr.groupby(meta[CELL_TYPE_COL]).mean()
print("Prototype matrix:", prototypes.shape, "(cell_types × genes)")
print(prototypes.index.tolist())

## 4 · L2-normalise for cosine similarity

In [ ]:
proto_norm = pd.DataFrame(
    normalize(prototypes.values, norm="l2"),
    index=prototypes.index,
    columns=prototypes.columns
)

## 5 · Save

In [ ]:
prototypes.to_csv(PROTOTYPES_PATH)
proto_norm.to_csv(PROTOTYPES_NORM_PATH)
print(f"Saved prototypes          → {PROTOTYPES_PATH}")
print(f"Saved prototypes_normalized → {PROTOTYPES_NORM_PATH}")